# L50 · MFCC 完整流水线 — 信号 → STFT → Mel → log → DCT，每步输出形状确认

**目标**：串联所有 DSP 步骤，手工实现完整 MFCC 提取，结果与 `aurora.audio.mfcc.mfcc()` 数值对齐。

🔗 Aurora 连接：`aurora.audio.mfcc.mfcc()`、`aurora.audio.mfcc.dct_ii()`、`aurora.audio.mel.mel_spectrogram()`；这套特征是 Month 2 关键词识别分类器的标准输入。

MFCC 把一帧音频压缩成 13 个数字：先用 mel 滤波器组模拟人耳的频率感知，再取对数压缩（log compression）动态范围，最后用 DCT 去相关（decorrelation）——得到的系数在统计上接近独立，对 GMM/神经网络分类器都很友好。整条流水线的每一步都有明确的物理动机，没有任何"魔法"。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from aurora.audio.io import sine

## 1. 公式全景

```
MFCC[n] = DCT-II( log( mel_filterbank @ |STFT|² ) )[:n_mfcc]
```

- `|STFT|²` 是每帧的**功率谱（power spectrum）**，形状 `(n_frames, n_fft//2+1)`
- `mel_filterbank @ |STFT|²` 将功率谱映射到 mel 频带，形状 `(n_frames, n_mels)`
- `log(...)` 压缩动态范围（幅度相差 100 倍 → dB 只差 20）
- `DCT-II` 去相关，系数从低到高对应从粗到细的谱包络信息
- 取前 `n_mfcc` 个系数（`[:n_mfcc]`）；k=0 是所有 mel 能量的直流均值，保留它与 librosa 默认行为一致

对于正交归一化（`norm="ortho"`）的 DCT-II，第 `k` 个系数为：

```
y[k] = w[k] * sum_{n=0}^{N-1} x[n] * cos( pi/N * (n + 0.5) * k )
```

其中 `w[0] = sqrt(1/N)`，`w[k>0] = sqrt(2/N)`。

In [ ]:
# 演示：DCT-II 去相关的效果
from aurora.audio.mfcc import dct_ii

sr = 16000
x = sine(440, duration=0.5, sample_rate=sr)

# 构造一帧 mel 对数谱（40 mel bins）
from aurora.audio.mel import mel_spectrogram, power_to_db
log_mel = power_to_db(mel_spectrogram(x, sr, n_mels=40, n_fft=1024, hop_length=256), top_db=None)
# 取第 5 帧示例
frame = log_mel[5]
dct_frame = dct_ii(frame)

fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].bar(range(len(frame)), frame)
axes[0].set_title("log-mel 谱（40 bins，第5帧）")
axes[0].set_xlabel("mel bin")
axes[0].set_ylabel("dB")
axes[1].bar(range(len(dct_frame)), dct_frame, color="orange")
axes[1].axvline(0.5, color="red", linestyle="--", label="k=0..12，共 n_mfcc=13 个（含 k=0）")
axes[1].set_title("DCT-II 系数（取 k=0..12，n_mfcc=13 含 k=0）")
axes[1].set_xlabel("DCT 系数索引 k")
axes[1].legend()
plt.tight_layout()
plt.show()
print(f"k=0（直流）: {dct_frame[0]:.3f}  k=1..4: {dct_frame[1:5].round(3)}")

## 2. 为什么 n_mfcc 通常取 13？

语音的谱包络可以用很少的参数描述：低阶 DCT 系数对应声道形状（元音特征），高阶系数对应细节（声门激励、噪声）。实验表明：

| n_mfcc | 适用场景 |
|--------|---------|
| 13 | 经典 MFCC，GMM-HMM 语音识别标准配置 |
| 20–40 | 深度学习分类器，信息量更足 |
| > 40 | 收益递减；等同于直接用 log-mel |

`n_mels` 控制 mel 滤波器组的分辨率（40–128 常见）；`n_mfcc` ≤ `n_mels`。增大 `n_mfcc` 相当于保留更多高阶谱细节，同时增加过拟合风险。

In [ ]:
# 演示：不同 n_mfcc 下的系数能量分布
from aurora.audio.mfcc import dct_ii
from aurora.audio.mel import mel_spectrogram, power_to_db

sr = 16000
x = sine(440, duration=1.0, sample_rate=sr)
log_mel = power_to_db(mel_spectrogram(x, sr, n_mels=40, n_fft=1024, hop_length=256), top_db=None)
all_coeffs = dct_ii(log_mel)  # (n_frames, 40)

energy_per_coeff = np.mean(all_coeffs**2, axis=0)  # 每个系数在所有帧上的均方值

plt.figure(figsize=(8, 3))
plt.bar(range(len(energy_per_coeff)), energy_per_coeff, color="steelblue")
plt.axvspan(-0.5, 0.5, color="red", alpha=0.2, label="k=0 跳过")
plt.axvspan(0.5, 13.5, color="green", alpha=0.15, label="k=1..13 (n_mfcc=13)")
plt.xlabel("DCT 系数索引 k")
plt.ylabel("平均能量（均方）")
plt.title("各 DCT 系数的平均能量——能量集中在低阶")
plt.legend()
plt.tight_layout()
plt.show()

## 3. 完整流程图

```
原始音频 x
  │
  ▼ STFT（汉宁窗，win_len=1024，hop=256）
功率谱 |X[k]|²   shape: (n_frames, n_fft//2+1)
  │
  ▼ mel_filterbank @   [mel矩阵，shape: (n_mels, n_fft//2+1)]
mel 能量           shape: (n_frames, n_mels)
  │
  ▼ log（power_to_db，top_db=None）
log-mel 谱         shape: (n_frames, n_mels)
  │
  ▼ DCT-II（ortho归一化）
DCT 系数           shape: (n_frames, n_mels)
  │
  ▼ 取 [:, :n_mfcc]    k=0..n_mfcc-1
MFCC               shape: (n_frames, n_mfcc)
```

每一步都是线性变换或逐元素操作，没有可学习参数。整条流水线的超参数只有 `n_fft`、`hop`、`n_mels`、`n_mfcc` 四个。

In [ ]:
# 演示：逐步走一遍流水线，打印每步形状
from aurora.audio.stft import power_spectrogram
from aurora.audio.mel import mel_filterbank, power_to_db
from aurora.audio.mfcc import dct_ii

sr = 16000
n_fft, hop, n_mels, n_mfcc = 1024, 256, 40, 13
x = sine(440, duration=1.0, sample_rate=sr)
print(f"x.shape           = {x.shape}")

power = power_spectrogram(x, n_fft=n_fft, hop_length=hop)
print(f"power.shape       = {power.shape}  ← (n_frames, n_fft//2+1)")

fb = mel_filterbank(n_mels, n_fft, sr)
mel = power @ fb.T
print(f"mel.shape         = {mel.shape}  ← (n_frames, n_mels)")

log_mel = power_to_db(mel, top_db=None)
print(f"log_mel.shape     = {log_mel.shape}")

coeffs = dct_ii(log_mel)
print(f"dct_coeffs.shape  = {coeffs.shape}")

mfcc_out = coeffs[:, :n_mfcc]
print(f"mfcc_out.shape    = {mfcc_out.shape}  ← (n_frames, n_mfcc) 最终输出")

## 4. ✏️ 实现 `my_mfcc(x, sr, n_mfcc=13, n_mels=40, win_len=1024, hop=256)`

**推理路线**：
1. 调用 `mel_spectrogram(x, sr, n_mels=n_mels, n_fft=win_len, hop_length=hop)` 得到 mel 功率谱，形状 `(n_frames, n_mels)`
2. 调用 `power_to_db(..., top_db=None)` 取对数，形状不变
3. 调用 `dct_ii(log_mel, norm="ortho")` 对每帧做 DCT，形状 `(n_frames, n_mels)`
4. 取 `coeffs[:, :n_mfcc]`——保留 k=0..n_mfcc-1，返回形状 `(n_frames, n_mfcc)`

**参考输入输出**：
```python
sr = 16000
x = sine(440, duration=1.0, sample_rate=sr)  # shape (16000,)
out = my_mfcc(x, sr, n_mfcc=13)
# out.shape == (63, 13)   # 具体帧数：16000//256+1 = 63
# out[0, :4] 与 aurora.audio.mfcc.mfcc(x, sr, n_mfcc=13, n_fft=1024, hop_length=256, n_mels=40) 数值一致
```

> 💡 需要提示？完成练习后可参考 `solutions/` 目录中的参考实现。


In [ ]:
def my_mfcc(x, sr, n_mfcc=13, n_mels=40, win_len=1024, hop=256):
    """MFCC 提取：x -> (n_frames, n_mfcc)"""
    from aurora.audio.mel import mel_spectrogram, power_to_db
    from aurora.audio.mfcc import dct_ii

    # ✏️ TODO 1: 计算 mel 功率谱，shape (n_frames, n_mels)
    mel = None

    # ✏️ TODO 2: 取对数（power_to_db，top_db=None），shape 不变
    log_mel = None

    # ✏️ TODO 3: 对每帧做 DCT-II（norm="ortho"），shape (n_frames, n_mels)
    coeffs = None

    # ✏️ TODO 4: 取前 n_mfcc 个系数：[:, :n_mfcc]，shape (n_frames, n_mfcc)
    return None

In [ ]:
from aurora.audio.mfcc import mfcc as aurora_mfcc

sr = 16000
x = sine(440, duration=1.0, sample_rate=sr)

out = my_mfcc(x, sr, n_mfcc=13, n_mels=40, win_len=1024, hop=256)

if out is None:
    print('⬜ my_mfcc 未实现，请完成四个 TODO 再运行')
else:
    ref = aurora_mfcc(x, sr, n_mfcc=13, n_fft=1024, hop_length=256, n_mels=40)
    print(f'my_mfcc shape : {out.shape}')
    print(f'aurora  shape : {ref.shape}')
    assert out.shape == ref.shape, f'形状不匹配: {out.shape} vs {ref.shape}'
    assert np.allclose(out, ref, atol=1e-5), '数值不一致，请检查 TODO 步骤'
    print('✅ 形状和数值均与 aurora.audio.mfcc.mfcc() 一致')


## 5. 参数实验：MFCC 热图

**实验目标**：可视化 13 个系数随时间的变化。

- **横轴**：帧编号（时间）
- **纵轴**：系数编号 1–13（越低 → 越粗粒度的谱形状）
- **预期现象**：
  - 系数 1（MFCC-1）变化相对平滑，代表整体音色包络
  - 系数 12–13 变化最快，携带细粒度谱纹理信息
  - 对纯正弦波，高阶系数接近零（频谱很纯净）；尝试换成 `chirp` 信号观察差异

修改 `n_mfcc=20` 后，图的纵轴延伸，可以看到 20 阶之后能量快速衰减。

In [ ]:
from aurora.audio.io import chirp

sr = 16000
x_chirp = chirp(f0=200, f1=4000, duration=2.0, sample_rate=sr)
mfcc_mat = my_mfcc(x_chirp, sr, n_mfcc=13, n_mels=40, win_len=1024, hop=256)

if mfcc_mat is None:
    print('⬜ 请先完成 Cell 10 的 my_mfcc TODO，再运行本格')
else:
    import matplotlib.pyplot as plt, numpy as np
    plt.figure(figsize=(10, 4))
    plt.imshow(
        mfcc_mat.T,
        aspect='auto', origin='lower', cmap='magma',
    )
    plt.colorbar(label='MFCC 值')
    plt.xlabel('帧编号'); plt.ylabel('系数编号 k')
    plt.title('Chirp 信号的 MFCC 热图（频率从 200→4000 Hz）')
    plt.tight_layout(); plt.show()
    print(f'✅ mfcc_mat.shape = {mfcc_mat.shape}')


## 本课收束

`my_mfcc()` 串联了 `mel_spectrogram → power_to_db → dct_ii → [:n_mfcc]` 四步，输出形状 `(n_frames, n_mfcc)` 的 float64 数组，数值与 `aurora.audio.mfcc.mfcc()` 完全对齐（atol=1e-5）。该函数是 `aurora.audio.mfcc` 模块的核心，Month 2 关键词识别分类器将直接以 `(n_frames, 13)` 的 MFCC 矩阵作为输入特征。下一课（L51）将在真实语音片段上提取 MFCC 并做简单的 GMM 分类实验，验证特征的区分能力。

## ✏️ 闭卷推导检查格 — MFCC 流水线维度追踪

**规则：关闭上方所有格，仅凭记忆完成以下推导。**

**题目**：给定参数：
- 信号长度  = 16000$（1秒，16kHz）
- {fft} = 1024$， = 256$（hop），{mel} = 40$，{mfcc} = 13$

逐步写出每一步输出的形状：

| 步骤 | 操作 | 输出形状 |
|------|------|---------|
| 1 | 信号 | $(L,)$ = (___,) |
| 2 | STFT → magnitude | (___,  ___) |
| 3 | Mel 滤波 | (___, ___) |
| 4 | log（dB） | (___, ___) |
| 5 | DCT-II → MFCC | (___, ___) |

（在此处填表...）

In [ ]:
# 验证：逐步输出形状与推导一致
import numpy as np, sys
sys.path.insert(0, 'src')
from aurora.audio.stft import stft
from aurora.audio.mel import mel_filterbank, power_to_db
from aurora.audio.mfcc import mfcc, dct_ii

SR, N_FFT, HOP, N_MELS, N_MFCC = 16000, 1024, 256, 40, 13
signal = np.sin(2*np.pi*440*np.arange(SR)/SR).astype(np.float32)

spec   = stft(signal, n_fft=N_FFT, hop_length=HOP)      # (n_frames, n_bins)
n_frames, n_bins = spec.shape
fb     = mel_filterbank(n_mels=N_MELS, n_fft=N_FFT, sample_rate=SR)
mel_e  = (np.abs(spec)**2) @ fb.T                        # (n_frames, N_MELS)
log_m  = power_to_db(mel_e)                              # (n_frames, N_MELS)
coeffs = mfcc(signal, sample_rate=SR, n_mfcc=N_MFCC,
              n_fft=N_FFT, hop_length=HOP, n_mels=N_MELS) # (n_frames, N_MFCC)

shapes = [spec.shape, mel_e.shape, log_m.shape, coeffs.shape]
labels = ['STFT', 'Mel', 'log-Mel', 'MFCC']
for l, s in zip(labels, shapes): print(f'  {l}: {s}')
assert coeffs.shape == (n_frames, N_MFCC), f'最终形状 {coeffs.shape} 不符预期'
print('✅ MFCC 流水线维度全部验证通过')